# Adult mosquito population

Use the sliders to display the `adults` variable at different times.

In [ ]:
from pathlib import Path

import cartopy.crs as ccrs
import ipywidgets as widgets
import matplotlib.pyplot as plt
import xarray as xr

DATA_PATH = Path(
    "/home/ecarreno/SSC-Projects/b_REPOSITORIES/org_ssciwr/onehealth-model-backend/"
    "data/out/Pratik_datalake/Mosquito_abundance_Global_2024.nc"
)

dataset = xr.open_dataset(DATA_PATH)
adults = dataset["adults"].transpose("time", "latitude", "longitude")
adults

## World

In [ ]:
def plot_world(time_index):
    data = adults.isel(time=time_index)
    date = str(data["time"].values)[:10]

    ax = plt.axes(projection=ccrs.Robinson())
    data.plot(
        ax=ax,
        x="longitude",
        y="latitude",
        transform=ccrs.PlateCarree(),
        cmap="viridis",
        robust=True,
    )
    ax.coastlines()
    ax.set_global()
    ax.set_title(f"Adult mosquito population — {date}")
    plt.show()


widgets.interact(
    plot_world,
    time_index=widgets.IntSlider(
        min=0, max=adults.sizes["time"] - 1, continuous_update=False
    ),
)

## Europe

In [ ]:
europe = adults.sel(longitude=slice(-25, 45), latitude=slice(34, 72))


def plot_europe(time_index):
    data = europe.isel(time=time_index)
    date = str(data["time"].values)[:10]

    ax = plt.axes(projection=ccrs.PlateCarree())
    data.plot(
        ax=ax,
        x="longitude",
        y="latitude",
        transform=ccrs.PlateCarree(),
        cmap="viridis",
        robust=True,
    )
    ax.coastlines()
    ax.set_extent([-25, 45, 34, 72], crs=ccrs.PlateCarree())
    ax.set_title(f"Adult mosquito population in Europe — {date}")
    plt.show()


widgets.interact(
    plot_europe,
    time_index=widgets.IntSlider(
        min=0, max=europe.sizes["time"] - 1, continuous_update=False
    ),
)

## ERA5-Land temperature — 1 September 2024

Global daily mean 2 m temperature for the selected date in degrees Celsius.
Although the variable metadata says K, the file history records `cdo subc,273.15`: the
values are already in °C and must not be converted again.
The following sections can run independently of the mosquito maps. Figures use a
7.2-inch width, DejaVu Sans fonts (9 pt text, 11 pt titles, 8 pt ticks), and
perceptually uniform color maps. PDF and 300 dpi PNG copies are saved in
`notebooks/figures/`. Missing grid cells are shown in light gray.


In [ ]:
from pathlib import Path

import xarray as xr

# Locate the repository when running from its root or the notebooks directory.
PROJECT_ROOT = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "data/in/Pratik_datalake").is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Run this notebook from the repository or its notebooks directory."
    )

ERA5_DIR = PROJECT_ROOT / "data/in/Pratik_datalake"
FIGURE_DIR = PROJECT_ROOT / "notebooks/figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
MAP_DATE = "2024-12-31"
PAPER_STYLE = {
    "font.family": "DejaVu Sans",
    "font.size": 9,
    "axes.titlesize": 11,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "axes.linewidth": 0.6,
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
}


def plot_era5_paper(data, title, colorbar_label, cmap_name, filename, vmin=None):
    with plt.rc_context(PAPER_STYLE):
        fig, ax = plt.subplots(
            figsize=(7.2, 4.2),
            subplot_kw={"projection": ccrs.PlateCarree()},
            layout="constrained",
        )
        ax.set_facecolor("#eeeeee")
        cmap = plt.get_cmap(cmap_name).copy()
        cmap.set_bad((0, 0, 0, 0))
        mesh = data.plot.pcolormesh(
            ax=ax,
            x="longitude",
            y="latitude",
            transform=ccrs.PlateCarree(),
            cmap=cmap,
            vmin=vmin,
            add_colorbar=False,
            add_labels=False,
            rasterized=True,
        )
        ax.coastlines(resolution="110m", color="#333333", linewidth=0.45)
        ax.set_global()
        ax.set_xticks(range(-180, 181, 60), crs=ccrs.PlateCarree())
        ax.set_yticks(range(-90, 91, 30), crs=ccrs.PlateCarree())
        ax.set_xlabel("longitudes")
        ax.set_ylabel("latitudes")
        ax.tick_params(width=0.5, length=3)
        ax.grid(linewidth=0.3, color="#777777", alpha=0.5, linestyle=":")
        ax.set_title(title, pad=10)
        colorbar = fig.colorbar(
            mesh,
            ax=ax,
            orientation="horizontal",
            shrink=0.8,
            pad=0.05,
            aspect=40,
        )
        colorbar.set_label(colorbar_label)
        colorbar.ax.tick_params(width=0.5, length=3)
        for extension in ("pdf", "png"):
            fig.savefig(FIGURE_DIR / f"{filename}.{extension}", bbox_inches="tight")
        plt.show()
    return fig


with xr.open_dataset(
    ERA5_DIR / "ERA5land_global_t2m_daily_0.5_2024.nc"
) as temperature_dataset:
    temperature_c = temperature_dataset["t2m"].sel(valid_time=MAP_DATE).load()
temperature_c.attrs = {"long_name": "2 m temperature", "units": "°C"}

temperature_figure = plot_era5_paper(
    temperature_c,
    "ERA5-Land 2 m temperature — 1 September 2024",
    "2 m temperature (°C)",
    "inferno",
    f"era5_temperature_{MAP_DATE}",
)

## ERA5-Land precipitation — 1 September 2024

Total precipitation for the selected daily record in millimetres. Although the
variable metadata says m, the file history records `cdo mulc,1000`: the values
are already in mm and must not be converted again. The color scale starts at zero and includes the full data range;
no percentile clipping is applied. The label uses mm because the file does not
explicitly document an accumulation interval. Run the temperature section first
to define the shared plotting function and figure style.


In [ ]:
with xr.open_dataset(
    ERA5_DIR / "ERA5land_global_tp_daily_0.5_2024.nc"
) as precipitation_dataset:
    precipitation_mm = precipitation_dataset["tp"].sel(time=MAP_DATE).load()
precipitation_mm.attrs = {"long_name": "Total precipitation", "units": "mm"}

precipitation_figure = plot_era5_paper(
    precipitation_mm,
    "ERA5-Land total precipitation — 1 September 2024",
    "Total precipitation (mm)",
    "viridis",
    f"era5_precipitation_{MAP_DATE}",
    vmin=0,
)

In [ ]:
temperature_c